# 03. 실험 묶음 학습 (각자 자기 묶음)

`GROUP`만 자기 묶음으로 바꿔 실행한다. A: 주 검정(Marqo, 13회) / B: ResNet-50(5회) / C: CLIP + 다른 테스트 주(9회).

끝난 실험은 드라이브에 `run_info.json`이 생기고, 다시 실행하면 건너뛴다. Colab이 끊기면 0번 셀부터 다시 실행하면 이어서 돈다.

In [ ]:
# @title 0. 공통 준비 (모든 노트북 동일)
import sys
if any(m.startswith("msrs_paper") for m in sys.modules):
    raise RuntimeError("이전에 불러온 코드가 메모리에 남아 있습니다. 메뉴 '런타임 → 세션 다시 시작' 후 이 셀부터 다시 실행하세요.")
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

REPO_URL = "https://github.com/yuhwani/multimodal-recsys.git"  # upstream에 머지되면 3sunghynix 주소로 변경
BRANCH = "main"
!rm -rf /content/repo && git clone -q -b {BRANCH} {REPO_URL} /content/repo
!pip -q install -r /content/repo/paper/requirements.txt
!cd /content/repo && git log -1 --format="코드 커밋: %h %s (%cd)"

import sys
sys.path.insert(0, "/content/repo/paper")
from msrs_paper.config import load_all, Paths
cfg, exps, plan = load_all()
DRIVE_ROOT = cfg["drive_root"]  # 공유 폴더 바로가기 경로가 다르면 이 줄만 바꾼다
paths = Paths(DRIVE_ROOT).make_dirs()
print("드라이브 폴더:", paths.root)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# @title 1. 공유 준비물 확인 (체크섬이 다르면 멈춤)
from msrs_paper.utils import verify_manifest
m = verify_manifest(paths)
print(f"준비물 {len(m['files'])}개 일치 — 준비물 커밋 {m['git_commit']}")

In [ ]:
# @title 2. 내 묶음 선택
GROUP = "A"  # @param ["A", "B", "C"]
assert cfg["train"]["tuned"], "02_tuning 결과를 paper.yaml에 반영하고 tuned: true로 커밋한 뒤 실행하세요"
assert plan["status"] == "final"
plan_runs = [(b["split"], c, s) for b in exps["groups"][GROUP] for s in b["seeds"] for c in b["conds"]]
done = [r for r in plan_runs if (paths.run_dir(GROUP, r[0], f"{r[1]}_seed{r[2]}") / "run_info.json").exists()]
print(f"묶음 {GROUP}: 전체 {len(plan_runs)}회, 완료 {len(done)}회")

In [ ]:
# @title 3. 학습 실행 (로그는 logs/ 에도 저장)
from msrs_paper import two_tower
from msrs_paper.utils import now
log_file = paths.logs / f"group{GROUP}.log"
def log(msg):
    print(msg)
    with open(log_file, "a", encoding="utf-8") as f:
        f.write(f"{now()} {msg}\n")
two_tower.run_group(cfg, exps, paths, GROUP, log=log)
log(f"묶음 {GROUP} 완료")